In [3]:
from ultralytics import YOLO, RTDETR
import datetime
import os
import logging
import csv
import cv2
import torch
import shutil

模型訓練

In [2]:
# 儲存路徑設定
project = './results/Training'
date = datetime.date.today()
name = f'{date} RTl_wound'
output_dir = os.path.join(project, name)
log_file = os.path.join(output_dir,"Training.log")
os.makedirs(output_dir,exist_ok=True)

In [3]:
# log設定
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_file),  
        logging.StreamHandler()        
    ]
)

In [4]:
logger = logging.getLogger()
logger.info(f"log儲存路徑: {log_file}")

2026-04-22 13:12:06 - log儲存路徑: ./results/Training/2026-04-22 RTl_wound/Training.log


In [5]:
if torch.cuda.is_available():
    print("CUDA 版本：", torch.version.cuda)
    print("PyTorch 使用的 CUDA 版本：", torch.version.cuda)
    print("GPU 數量：", torch.cuda.device_count())
    print("目前使用的 GPU 名稱：", torch.cuda.get_device_name(0))
    print("GPU 記憶體使用狀況：")
    print(torch.cuda.memory_summary(device=0))
else:
    print("❌ CUDA 不可用，請檢查驅動或 CUDA 工具包安裝。")

CUDA 版本： 13.0
PyTorch 使用的 CUDA 版本： 13.0
GPU 數量： 1
目前使用的 GPU 名稱： NVIDIA GeForce RTX 5090
GPU 記憶體使用狀況：
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |   

In [ ]:
# 傷口部位檢測
try:
    logger.info("開始進行模型訓練")
    #model = YOLO("yolo26n.pt")
    model = RTDETR("rtdetr-l.pt")
    model.train(
        data='data/Wound2cutmix.yaml',
        epochs=900,
        batch=16,
        imgsz=640,
        project=project,
        name=name,
        exist_ok=True,
        # --- 以下為訓練加速參數 ---
        workers=4,       # 根據 CPU 核心數調整 (預設通常為 8)
        cache=True,      # 將圖片快取至 RAM，加速 IO (若 RAM 不夠請設為 False 或 'disk')
        amp=True,        # 開啟混合精度訓練 (FP16 + FP32)
        #device=0,        # 指定 GPU (若有多張 GPU 可寫 device=[0, 1, 2, 3])
        patience=100,   # 可選：加入 EarlyStopping 耐心值，提早結束無效訓練
        # save_period=1  # 每個Epoch儲存一次權重
    )
    # 將結果保存為onnx格式
    model.export(format='onnx', imgsz=640, half=True)
    logger.info("訓練完成，模型已保存為 ONNX 格式")
except KeyboardInterrupt:
    # 手動中斷訓練（如 Ctrl+C）
    logger.warning("訓練被手動中斷。")
except Exception as e:
    # 判斷是否是提早結束
    if "EarlyStopping" in str(e):
        logger.info(f"訓練已提早結束: {e}")
    else:
        # 捕捉其他未知錯誤
        logger.error(f"訓練過程中發生錯誤: {e}")

logger.info("日誌紀錄結束。")

2026-04-22 13:12:06 - 開始進行模型訓練


New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.30 🚀 Python-3.13.11 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32103MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/Wound2cutmix.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=900, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=2026-

/home/hank/anaconda3/envs/wound/lib/python3.13/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/900      12.3G     0.6253      1.508     0.3318         27        640: 100% ━━━━━━━━━━━━ 347/347 3.6it/s 1:36<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 10.5it/s 5.0s0.1s
                   all       1646       2625      0.829      0.711       0.79      0.548

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/900      13.2G     0.5837     0.5112     0.2424         28        640: 0% ──────────── 0/347  0.3s

/home/hank/anaconda3/envs/wound/lib/python3.13/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/900      13.2G      0.502     0.5057     0.2304         20        640: 100% ━━━━━━━━━━━━ 347/347 4.2it/s 1:22<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 11.7it/s 4.4s0.2s
                   all       1646       2625      0.862      0.723      0.805      0.522

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/900      13.2G     0.4284     0.5423     0.2086         25        640: 0% ──────────── 0/347  0.2s

... (164 outputs left out)

/home/hank/anaconda3/envs/wound/lib/python3.13/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     85/900      13.2G     0.4483     0.3862     0.1265         26        640: 100% ━━━━━━━━━━━━ 347/347 4.6it/s 1:16<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 11.2it/s 4.7s.2ss
                   all       1646       2625      0.898      0.719      0.793      0.542

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     86/900      13.2G     0.5114     0.3914     0.1265         31        640: 0% ──────────── 0/347  0.4s

/home/hank/anaconda3/envs/wound/lib/python3.13/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     86/900      13.2G     0.4731     0.3799     0.1325         28        640: 45% ━━━━━─────── 156/347 4.5it/s 35.3s<42.6s

測試集運行

In [11]:
# 定義測試結果路徑
testResult_Dir = f'./results/testing/yolo26s_wound' #結果儲存路徑
predictionsResult_Dir = os.path.join(testResult_Dir,'predictions')
pictureResult_Dir = os.path.join(testResult_Dir,'cropped_picture')
os.makedirs(predictionsResult_Dir,exist_ok =True)
os.makedirs(pictureResult_Dir,exist_ok =True)
csv_dir = os.path.join(testResult_Dir,'csv')
os.makedirs(csv_dir, exist_ok=True)
output_file = os.path.join(csv_dir, 'performance_metrics.csv')
print('資料夾建立完成!')

資料夾建立完成!


In [ ]:
import os
import glob

print("date =", date)
print("results exists:", os.path.exists("./results"))
print("Training exists:", os.path.exists("./results/Training"))

print("\nTraining subfolders:")
if os.path.exists("./results/Training"):
    print(os.listdir("./results/Training"))

print("\nAll best.pt:")
for p in glob.glob("./results/**/best.pt", recursive=True):
    print(p)

In [7]:
model = YOLO(f'./bestyolo26/yolo26s/best.pt')
print('Model Load Successfully! ')
print(model.model)


Model Load Successfully! 
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): Si

In [8]:
metrics = model.val(
    data='data/Wound2cutmix.yaml',
    split='test',
    batch=8,
    imgsz=640,
    device=0,
    save=True # save results
)
default_save_dir = metrics.save_dir

Ultralytics 8.4.30 🚀 Python-3.13.11 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32102MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 373.7±319.0 MB/s, size: 680.7 KB)
val: Scanning /media/hank/Disk/Experiment/Wound/Wound_detection/YOLOv10n/data/Wound2cutmix/labels/test.cache... 97 images, 36 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 133/133 37.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 25.7it/s 0.7s0.3s
                   all        133        130      0.819      0.546       0.69      0.383
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /media/hank/Disk/Experiment/Wound/Wound_detection/YOLOv10n/runs/detect/val5


In [ ]:
metrics_dict = metrics.results_dict

# 列印性能指標
print("\n性能指標:")
for key, value in metrics_dict.items():
    print(f"{key}: {value}")

In [ ]:
# 將測試的結果保存
with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Metric', 'Value'])
    for key, value in metrics_dict.items():
        writer.writerow([key, value])
print(f"\n性能指标已保存到: {output_file}")

In [ ]:
# 移動到指定位置
if os.path.exists(default_save_dir):
    for file_name in os.listdir(default_save_dir):
        full_file_name = os.path.join(default_save_dir, file_name)
        if os.path.isfile(full_file_name):
            shutil.move(full_file_name, os.path.join(predictionsResult_Dir, file_name))
    print(f"已將測試結果移動至: {predictionsResult_Dir}")
else:
    print(f"結果仍然保存於: {default_save_dir}")

保存bounding box結果(測試集)

In [5]:
imageDir = "./data/Wound2cutmix/images/test"
print(imageDir)

./data/Wound2cutmix/images/test


In [6]:
image_files = [f for f in os.listdir(imageDir) if f.endswith(('.jpg', '.png', '.jpeg'))]
if not image_files:
    print("未找到圖片")

In [12]:
for image_name in image_files:
    image_path = os.path.join(imageDir, image_name)

    results = model(image_path)

    # 读取原始图片
    image = cv2.imread(image_path)
    if image is None:
        print(f"無法載入: {image_path}")
        continue

    height, width, _ = image.shape

    # 5. 裁剪并保存每个目标
    for i, box in enumerate(results[0].boxes.xyxy):
        x1, y1, x2, y2 = map(int, box)  # 获取 bounding box 坐标
        class_id = int(results[0].boxes.cls[i])  # 获取类别 ID

        # 获取类别名称（若 `model.names` 不存在，使用默认名称）
        class_name = model.names[class_id] if hasattr(model, 'names') else f"class_{class_id}"

        # 按类别创建存储文件夹
        class_dir = os.path.join(pictureResult_Dir, f"{class_name}")
        os.makedirs(class_dir, exist_ok=True)

        # 裁剪对象区域
        cropped = image[y1:y2, x1:x2]
        if cropped.size == 0:
            print(f"无效裁剪区域: {image_name} ({x1}, {y1}, {x2}, {y2})")
            continue

        # 生成保存路径
        cropped_filename = f"{os.path.splitext(image_name)[0]}_class{class_id}_{x1}_{y1}.jpg"
        cropped_filepath = os.path.join(class_dir, cropped_filename)

        # 保存裁剪后的图片
        cv2.imwrite(cropped_filepath, cropped)
        print(f"已保存: {cropped_filepath}")


image 1/1 /media/hank/Disk/Experiment/Wound/Wound_detection/YOLOv10n/data/Wound2cutmix/images/test/2-025-P3_5_WCC_2_1.jpg: 480x640 1 Wound, 3.6ms
Speed: 1.2ms preprocess, 3.6ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
已保存: ./results/testing/yolo26s_wound/cropped_picture/Wound/2-025-P3_5_WCC_2_1_class0_437_256.jpg

image 1/1 /media/hank/Disk/Experiment/Wound/Wound_detection/YOLOv10n/data/Wound2cutmix/images/test/2-042-P4_6_WCC_4_1.jpg: 640x480 1 Wound, 15.5ms
Speed: 1.8ms preprocess, 15.5ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 480)
已保存: ./results/testing/yolo26s_wound/cropped_picture/Wound/2-042-P4_6_WCC_4_1_class0_967_579.jpg

image 1/1 /media/hank/Disk/Experiment/Wound/Wound_detection/YOLOv10n/data/Wound2cutmix/images/test/WCC-2-52.jpg: 480x640 1 Wound, 3.6ms
Speed: 0.9ms preprocess, 3.6ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)
已保存: ./results/testing/yolo26s_wound/cropped_picture/Wound/WCC-2-52_class0_309_190.jp